# VocaLip Thai — Experiment 0 Recovery Notebook

**Supervised Mouth-only Word-Level Baseline — 44 Thai words**

Notebook นี้เขียนใหม่สำหรับกู้ Experiment 0 หลัง kernel reset โดยใช้ไฟล์ที่สร้างไว้แล้ว:

- clean manifest: `word_train_manifest_v5_baseline_clean.csv`
- mouth cache: `mouth_cache_v3`
- best checkpoint: `01_word_level_baseline_best.pt`

**Notebook นี้ไม่ preprocess และไม่ train ใหม่โดยอัตโนมัติ**  
เป้าหมายคือ restore model → evaluate test set → diagnostics → export final record

> Evaluation scope: **in-domain closed-vocabulary baseline**, not speaker-disjoint unseen-patient evaluation.


## Pipeline

1. Environment / imports  
2. Experiment config  
3. Reproducibility  
4. Load clean dataset  
5. Dataset audit  
6. Mouth-cache dataset  
7. Class weights + DataLoaders  
8. Original baseline architecture  
9. Restore best checkpoint  
10. Metrics helpers (NumPy; no scikit-learn required)  
11. Final test inference  
12. Per-class report  
13. Confusion matrix  
14. Confusion-pair analysis  
15. Prediction inspection  
16. Final experiment export  


In [ ]:
# ============================================================
# CELL 1 : ENVIRONMENT / IMPORTS
# ============================================================

import os
import sys
import json
import math
import random
import time
import hashlib
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE

print("Device :", DEVICE)
print("scikit-learn required: NO")


In [ ]:
# ============================================================
# CELL 2 : EXPERIMENT CONFIG
# ============================================================

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

MANIFEST_PATH = ROOT / "word_train_manifest_v5_baseline_clean.csv"
LABEL_MAP_PATH = ROOT / "word_label_map_v5.json"
CACHE_ROOT = ROOT / "mouth_cache_v3"

MODEL_DIR = ROOT / "models_01_baseline"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "01_word_level_baseline_best.pt"
LAST_MODEL_PATH = MODEL_DIR / "01_word_level_baseline_last.pt"
HISTORY_CSV_PATH = MODEL_DIR / "01_word_level_baseline_history.csv"

NUM_CLASSES = 44
TARGET_FRAMES = 16
ROI_SIZE = 112
CHANNELS = 3

FEATURE_DIM = 128
GRU_HIDDEN = 256
GRU_LAYERS = 2
MODEL_DROPOUT = 0.20

BATCH_SIZE = 16
NUM_WORKERS = 0

LR = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 40
PATIENCE = 8
GRAD_CLIP_NORM = 5.0

USE_CLASS_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False

CLASS_WEIGHT_MIN = 0.25
CLASS_WEIGHT_MAX = 4.0

EXPERIMENT_ID = "EXP0"
EXPERIMENT_NAME = "Supervised Mouth-only Word-Level Baseline"

# Final reference values from the completed Experiment 0.
# These are ONLY for recovery verification — do not tune using them.
EXP0_REFERENCE_BEST_EPOCH = 36
EXP0_REFERENCE_VAL_ACCURACY = 0.2542
EXP0_REFERENCE_VAL_MACRO_F1 = 0.2289
EXP0_REFERENCE_TEST_LOSS = 2.748788
EXP0_REFERENCE_TEST_ACCURACY = 0.2625
EXP0_REFERENCE_TEST_MACRO_F1 = 0.2327
EXP0_REFERENCE_TEST_TOP3 = 0.4826

print("=" * 80)
print("EXPERIMENT CONFIG")
print("=" * 80)
print("ROOT       :", ROOT)
print("Manifest   :", MANIFEST_PATH)
print("Cache      :", CACHE_ROOT)
print("Checkpoint :", BEST_MODEL_PATH)
print("Classes    :", NUM_CLASSES)
print("Frames     :", TARGET_FRAMES)
print("ROI        :", ROI_SIZE)
print("Batch      :", BATCH_SIZE)
print("Device     :", DEVICE)

assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert CACHE_ROOT.exists(), f"Missing cache root: {CACHE_ROOT}"
assert BEST_MODEL_PATH.exists(), f"Missing checkpoint: {BEST_MODEL_PATH}"


In [ ]:
# ============================================================
# CELL 3 : REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


In [ ]:
# ============================================================
# CELL 4 : LOAD CLEAN DATASET
# ============================================================

df = pd.read_csv(MANIFEST_PATH)

required_columns = {"split", "model_class_id", "word"}
missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Missing manifest columns: {missing_columns}"

df["model_class_id"] = df["model_class_id"].astype(int)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

class_word_table = (
    df[["model_class_id", "word"]]
    .drop_duplicates()
    .sort_values("model_class_id")
    .reset_index(drop=True)
)

assert len(class_word_table) == NUM_CLASSES

id_to_word = {
    int(row.model_class_id): str(row.word)
    for row in class_word_table.itertuples(index=False)
}
word_to_id = {word: cid for cid, word in id_to_word.items()}

print("=" * 80)
print("CLEAN DATASET")
print("=" * 80)
print("Total :", len(df))
print("Train :", len(train_df))
print("Val   :", len(val_df))
print("Test  :", len(test_df))
print("Classes:", df["model_class_id"].nunique())

assert len(df) == 2245
assert len(train_df) == 1632
assert len(val_df) == 354
assert len(test_df) == 259
assert df["model_class_id"].nunique() == 44

print("\nFirst 10 labels:")
for i in range(10):
    print(f"{i:2d} -> {id_to_word[i]}")


In [ ]:
# ============================================================
# CELL 5 : DATASET AUDIT
# ============================================================

split_summary = (
    df.groupby("split")
      .agg(
          samples=("model_class_id", "size"),
          classes=("model_class_id", "nunique")
      )
      .reset_index()
)

display(split_summary)

train_counts = train_df["model_class_id"].value_counts().sort_index()
val_counts   = val_df["model_class_id"].value_counts().sort_index()
test_counts  = test_df["model_class_id"].value_counts().sort_index()

audit_table = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "word": [id_to_word[i] for i in range(NUM_CLASSES)],
    "train": [int(train_counts.get(i, 0)) for i in range(NUM_CLASSES)],
    "val": [int(val_counts.get(i, 0)) for i in range(NUM_CLASSES)],
    "test": [int(test_counts.get(i, 0)) for i in range(NUM_CLASSES)],
})

display(audit_table)

assert (audit_table[["train", "val", "test"]] > 0).all().all()

print("=" * 80)
print("DATASET AUDIT PASSED")
print("=" * 80)
print("Train min/max:", audit_table["train"].min(), audit_table["train"].max())
print("Val   min/max:", audit_table["val"].min(), audit_table["val"].max())
print("Test  min/max:", audit_table["test"].min(), audit_table["test"].max())
print()
print("Scope reminder: current evaluation is IN-DOMAIN and NOT speaker-disjoint.")


In [ ]:
# ============================================================
# CELL 6 : CACHE PATH RESOLVER + DATASET
# ============================================================

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32
).view(1, 3, 1, 1)


def resolve_cache_path(row):
    # Prefer paths already stored in manifest.
    for key in ["mouth_cache_path", "cache_path"]:
        if key in row.index and pd.notna(row[key]):
            p = Path(str(row[key]))
            if p.exists():
                return p

    # Common identifier-based fallbacks.
    for key in [
        "clip_id",
        "segment_id",
        "sample_id",
        "video_id",
        "filename",
        "clip_name"
    ]:
        if key in row.index and pd.notna(row[key]):
            stem = Path(str(row[key])).stem
            for p in [
                CACHE_ROOT / f"{stem}.npz",
                CACHE_ROOT / stem / "mouth.npz",
            ]:
                if p.exists():
                    return p

    # Video stem fallback.
    if "video_path" in row.index and pd.notna(row["video_path"]):
        stem = Path(str(row["video_path"])).stem
        p = CACHE_ROOT / f"{stem}.npz"
        if p.exists():
            return p

    raise FileNotFoundError(
        "Cannot resolve mouth cache path for row:\n"
        + str(row.to_dict())
    )


class VocaLipWordDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = resolve_cache_path(row)

        with np.load(cache_path) as data:
            frames = data["frames"].copy()

        assert frames.shape == (
            TARGET_FRAMES,
            ROI_SIZE,
            ROI_SIZE,
            3
        ), f"Unexpected shape {frames.shape}: {cache_path}"

        frames = torch.from_numpy(frames).float() / 255.0
        frames = frames.permute(0, 3, 1, 2)  # T,H,W,C -> T,C,H,W
        frames = (frames - IMAGENET_MEAN) / IMAGENET_STD

        label = int(row["model_class_id"])

        return frames, label


train_dataset = VocaLipWordDataset(train_df)
val_dataset   = VocaLipWordDataset(val_df)
test_dataset  = VocaLipWordDataset(test_df)

x0, y0 = test_dataset[0]

print("Sample shape:", tuple(x0.shape))
print("Sample label:", y0, id_to_word[y0])

assert x0.shape == (TARGET_FRAMES, 3, ROI_SIZE, ROI_SIZE)


In [ ]:
# ============================================================
# CELL 7 : CLASS WEIGHTS + DATALOADERS
# ============================================================

train_counts = train_df["model_class_id"].value_counts().sort_index()

class_weights_np = np.zeros(NUM_CLASSES, dtype=np.float32)

for class_id in range(NUM_CLASSES):
    count = int(train_counts.loc[class_id])
    class_weights_np[class_id] = len(train_df) / (NUM_CLASSES * count)

class_weights_np = np.clip(
    class_weights_np,
    CLASS_WEIGHT_MIN,
    CLASS_WEIGHT_MAX
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # recovery/evaluation notebook
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("Class weight min :", class_weights_np.min())
print("Class weight max :", class_weights_np.max())
print("Class weight mean:", class_weights_np.mean())
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

xb, yb = next(iter(test_loader))
print("Batch:", tuple(xb.shape), tuple(yb.shape))


In [ ]:
# ============================================================
# CELL 8 : ORIGINAL BASELINE MODEL
# ============================================================

class FrameCNN(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 96, 3, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )

        # Linear must remain projection[1] to match saved checkpoint.
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, feature_dim)
        )

    def forward(self, x):
        return self.projection(self.features(x))


class TemporalAttention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        scores = self.attention(x).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return context, weights


class VocaLipWordModel(nn.Module):
    def __init__(
        self,
        num_classes=44,
        feature_dim=128,
        gru_hidden=256,
        gru_layers=2,
        dropout=0.20
    ):
        super().__init__()

        self.frame_encoder = FrameCNN(feature_dim)

        self.temporal_encoder = nn.GRU(
            input_size=feature_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(dropout if gru_layers > 1 else 0.0)
        )

        temporal_dim = gru_hidden * 2

        self.temporal_attention = TemporalAttention(temporal_dim)

        self.classifier = nn.Sequential(
            nn.LayerNorm(temporal_dim),
            nn.Dropout(dropout),
            nn.Linear(temporal_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.reshape(B * T, C, H, W)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(B, T, -1)

        temporal_features, _ = self.temporal_encoder(frame_features)
        context, attention_weights = self.temporal_attention(temporal_features)

        logits = self.classifier(context)

        return logits, attention_weights


model = VocaLipWordModel(
    num_classes=NUM_CLASSES,
    feature_dim=FEATURE_DIM,
    gru_hidden=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=MODEL_DROPOUT
).to(DEVICE)

total_parameters = sum(p.numel() for p in model.parameters())

print("Parameters:", f"{total_parameters:,}")
assert total_parameters == 2187405


In [ ]:
# ============================================================
# CELL 9 : RESTORE BEST CHECKPOINT
# ============================================================

try:
    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=False
    )
except TypeError:
    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE
    )

state_dict = None

for key in ["model_state_dict", "state_dict", "model"]:
    if key in checkpoint and isinstance(checkpoint[key], dict):
        state_dict = checkpoint[key]
        print("Checkpoint state key:", key)
        break

if state_dict is None:
    if sum(torch.is_tensor(v) for v in checkpoint.values()) > 0:
        state_dict = checkpoint
        print("Checkpoint is raw state_dict")

assert state_dict is not None

model.load_state_dict(state_dict, strict=True)
model.eval()

checkpoint_epoch = checkpoint.get("epoch", None)
checkpoint_val_metrics = checkpoint.get("val_metrics", None)
checkpoint_best_f1 = checkpoint.get("best_val_macro_f1", None)

xb, yb = next(iter(test_loader))
xb = xb.to(DEVICE)

with torch.inference_mode():
    logits, attn = model(xb)

print("=" * 80)
print("CHECKPOINT RESTORE")
print("=" * 80)
print("Checkpoint epoch:", checkpoint_epoch)
print("Val metrics     :", checkpoint_val_metrics)
print("Best val F1     :", checkpoint_best_f1)
print("Logits          :", tuple(logits.shape))
print("Attention       :", tuple(attn.shape))
print("Strict load     : OK")
print("Forward pass    : OK")
print("Training        : NOT RUN")


In [ ]:
# ============================================================
# CELL 10 : METRICS HELPERS — NO SCIKIT-LEARN
# ============================================================

def accuracy_numpy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))


def per_class_metrics_numpy(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rows = []

    for class_id in range(num_classes):
        tp = int(np.sum((y_true == class_id) & (y_pred == class_id)))
        fp = int(np.sum((y_true != class_id) & (y_pred == class_id)))
        fn = int(np.sum((y_true == class_id) & (y_pred != class_id)))
        support = int(np.sum(y_true == class_id))
        predicted_count = int(np.sum(y_pred == class_id))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        if (precision + recall) > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        rows.append({
            "class_id": class_id,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": support,
            "correct": tp,
            "predicted_count": predicted_count,
        })

    return pd.DataFrame(rows)


def macro_f1_numpy(y_true, y_pred, num_classes):
    report = per_class_metrics_numpy(y_true, y_pred, num_classes)
    return float(report["f1"].mean())


def confusion_matrix_numpy(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1

    return cm


print("NumPy metrics helpers: READY")


In [ ]:
# ============================================================
# CELL 11 : FINAL TEST INFERENCE
# ============================================================

criterion = nn.CrossEntropyLoss(weight=class_weights)

all_targets = []
all_predictions = []
all_probabilities = []
all_top3_predictions = []
all_top3_probabilities = []
all_confidences = []
all_attention_weights = []

total_loss = 0.0
total_samples = 0

start_time = time.time()

model.eval()

with torch.inference_mode():
    for frames, labels in tqdm(test_loader, desc="Test inference"):
        frames = frames.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits, attention_weights = model(frames)
        loss = criterion(logits, labels)

        bs = int(labels.size(0))
        total_loss += float(loss.item()) * bs
        total_samples += bs

        probabilities = torch.softmax(logits, dim=1)
        confidences, predictions = probabilities.max(dim=1)
        top3_probabilities, top3_predictions = torch.topk(
            probabilities,
            k=3,
            dim=1
        )

        all_targets.append(labels.cpu())
        all_predictions.append(predictions.cpu())
        all_probabilities.append(probabilities.cpu())
        all_top3_predictions.append(top3_predictions.cpu())
        all_top3_probabilities.append(top3_probabilities.cpu())
        all_confidences.append(confidences.cpu())
        all_attention_weights.append(attention_weights.cpu())

test_targets = torch.cat(all_targets).numpy()
test_predictions = torch.cat(all_predictions).numpy()
test_probabilities = torch.cat(all_probabilities).numpy()
test_top3_predictions = torch.cat(all_top3_predictions).numpy()
test_top3_probabilities = torch.cat(all_top3_probabilities).numpy()
test_confidences = torch.cat(all_confidences).numpy()
test_attention_weights = torch.cat(all_attention_weights).numpy()

test_loss = float(total_loss / total_samples)
test_accuracy = accuracy_numpy(test_targets, test_predictions)
test_macro_f1 = macro_f1_numpy(
    test_targets,
    test_predictions,
    NUM_CLASSES
)

true_in_top3 = np.any(
    test_top3_predictions == test_targets[:, None],
    axis=1
)

test_top3_accuracy = float(true_in_top3.mean())

correct_mask = test_targets == test_predictions
wrong_mask = ~correct_mask

num_correct = int(correct_mask.sum())
num_wrong = int(wrong_mask.sum())
num_true_in_top3 = int(true_in_top3.sum())
wrong_true_in_top3 = int((wrong_mask & true_in_top3).sum())

wrong_true_in_top3_rate = (
    wrong_true_in_top3 / num_wrong
    if num_wrong > 0 else 0.0
)

sorted_indices = np.argsort(-test_probabilities, axis=1)
test_true_ranks = np.zeros(total_samples, dtype=np.int64)

for i in range(total_samples):
    position = np.where(
        sorted_indices[i] == test_targets[i]
    )[0]
    test_true_ranks[i] = int(position[0] + 1)

test_metrics = {
    "loss": test_loss,
    "accuracy": test_accuracy,
    "macro_f1": test_macro_f1,
    "top3_accuracy": test_top3_accuracy,
    "num_samples": total_samples,
    "num_correct": num_correct,
    "num_wrong": num_wrong,
}

baseline_test_results = {
    "experiment": EXPERIMENT_ID,
    "name": EXPERIMENT_NAME,
    "checkpoint": str(BEST_MODEL_PATH),
    "num_classes": NUM_CLASSES,
    "num_test_samples": total_samples,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_top3_accuracy": test_top3_accuracy,
    "correct": num_correct,
    "wrong": num_wrong,
    "true_in_top3": num_true_in_top3,
    "wrong_top1_true_in_top3": wrong_true_in_top3,
    "wrong_top1_true_in_top3_rate": wrong_true_in_top3_rate,
}

elapsed = time.time() - start_time

print("=" * 88)
print("FINAL TEST")
print("=" * 88)
print(f"Samples      : {total_samples}")
print(f"Loss         : {test_loss:.6f}")
print(f"Accuracy     : {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"Macro F1     : {test_macro_f1:.4f}")
print(f"Top-3        : {test_top3_accuracy:.4f} ({test_top3_accuracy:.2%})")
print(f"Correct      : {num_correct}/{total_samples}")
print(f"Wrong        : {num_wrong}/{total_samples}")
print(
    f"Wrong Top-1 but true Top-3: "
    f"{wrong_true_in_top3}/{num_wrong} "
    f"({wrong_true_in_top3_rate:.2%} of errors)"
)
print(f"Inference sec: {elapsed:.2f}")

match = (
    abs(test_accuracy - EXP0_REFERENCE_TEST_ACCURACY) < 0.001
    and abs(test_macro_f1 - EXP0_REFERENCE_TEST_MACRO_F1) < 0.001
    and abs(test_top3_accuracy - EXP0_REFERENCE_TEST_TOP3) < 0.001
)

print()
if match:
    print("✅ Recovery matches completed Experiment 0.")
else:
    print("⚠️ Result differs from the completed Experiment 0.")
    print("Do NOT retrain yet; inspect preprocessing/evaluation consistency first.")


In [ ]:
# ============================================================
# CELL 12 : PER-CLASS CLASSIFICATION REPORT
# ============================================================

class_report_df = per_class_metrics_numpy(
    test_targets,
    test_predictions,
    NUM_CLASSES
)

class_report_df["word"] = class_report_df["class_id"].map(id_to_word)

class_report_df = class_report_df[
    [
        "class_id",
        "word",
        "precision",
        "recall",
        "f1",
        "support",
        "correct",
        "predicted_count",
    ]
]

report_by_f1 = class_report_df.sort_values(
    ["f1", "support"],
    ascending=[False, False]
).reset_index(drop=True)

zero_recall_df = class_report_df[
    class_report_df["recall"] == 0
].copy()

print("=" * 88)
print("PER-CLASS REPORT")
print("=" * 88)
print("Test samples       :", len(test_targets))
print("Classes            :", NUM_CLASSES)
print("Accuracy           :", f"{test_accuracy:.4f}")
print("Macro F1           :", f"{test_macro_f1:.4f}")
print("Zero-recall classes:", len(zero_recall_df))

display(class_report_df.round(4))

print("\nTOP 10 BY F1")
display(report_by_f1.head(10).round(4))

print("\nZERO-RECALL CLASSES")
display(zero_recall_df[["class_id", "word", "support"]])

CLASS_REPORT_PATH = MODEL_DIR / "01_word_level_baseline_test_class_report.csv"

class_report_df.to_csv(
    CLASS_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", CLASS_REPORT_PATH)


In [ ]:
# ============================================================
# CELL 13 : CONFUSION MATRIX
# ============================================================

cm_raw = confusion_matrix_numpy(
    test_targets,
    test_predictions,
    NUM_CLASSES
)

row_sum = cm_raw.sum(axis=1, keepdims=True)

cm_normalized = np.divide(
    cm_raw,
    row_sum,
    out=np.zeros_like(cm_raw, dtype=np.float64),
    where=row_sum != 0
)

class_words = [id_to_word[i] for i in range(NUM_CLASSES)]

cm_raw_df = pd.DataFrame(
    cm_raw,
    index=class_words,
    columns=class_words
)

cm_normalized_df = pd.DataFrame(
    cm_normalized,
    index=class_words,
    columns=class_words
)

baseline_cm_raw = cm_raw
baseline_cm_normalized = cm_normalized

diagonal_correct = int(np.trace(cm_raw))
matrix_accuracy = diagonal_correct / int(cm_raw.sum())

print("=" * 88)
print("CONFUSION MATRIX")
print("=" * 88)
print("Diagonal correct:", diagonal_correct)
print("Matrix accuracy :", f"{matrix_accuracy:.4f}")

assert np.isclose(matrix_accuracy, test_accuracy)

fig, ax = plt.subplots(figsize=(18, 16))
im = ax.imshow(cm_normalized, interpolation="nearest", aspect="auto")
ax.set_title(
    "VocaLip Thai — Row-Normalized Confusion Matrix\n"
    "Rows=True | Columns=Predicted"
)
ax.set_xlabel("Predicted Word")
ax.set_ylabel("True Word")
ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))
ax.set_xticklabels(class_words, rotation=90, fontsize=8)
ax.set_yticklabels(class_words, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 14 : CONFUSION PAIR ANALYSIS
# ============================================================

class_support = cm_raw.sum(axis=1)

directional_rows = []

for true_id in range(NUM_CLASSES):
    support = int(class_support[true_id])

    for pred_id in range(NUM_CLASSES):
        if true_id == pred_id:
            continue

        count = int(cm_raw[true_id, pred_id])

        if count <= 0:
            continue

        directional_rows.append({
            "true_class_id": true_id,
            "true_word": id_to_word[true_id],
            "pred_class_id": pred_id,
            "pred_word": id_to_word[pred_id],
            "confusion_count": count,
            "true_support": support,
            "confusion_rate": count / support,
        })

directional_confusions_df = (
    pd.DataFrame(directional_rows)
    .sort_values(
        ["confusion_count", "confusion_rate", "true_support"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

pair_rows = []

for a in range(NUM_CLASSES):
    for b in range(a + 1, NUM_CLASSES):
        a_to_b = int(cm_raw[a, b])
        b_to_a = int(cm_raw[b, a])
        total = a_to_b + b_to_a

        if total <= 0:
            continue

        support_pair = int(class_support[a] + class_support[b])

        pair_rows.append({
            "class_a": a,
            "word_a": id_to_word[a],
            "class_b": b,
            "word_b": id_to_word[b],
            "a_to_b": a_to_b,
            "b_to_a": b_to_a,
            "pair_confusion_count": total,
            "pair_confusion_rate": (
                total / support_pair
                if support_pair > 0 else 0.0
            ),
            "bidirectional_min": min(a_to_b, b_to_a),
        })

pair_confusions_df = (
    pd.DataFrame(pair_rows)
    .sort_values(
        ["pair_confusion_count", "bidirectional_min", "pair_confusion_rate"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

bidirectional_confusions_df = (
    pair_confusions_df[
        pair_confusions_df["bidirectional_min"] > 0
    ]
    .reset_index(drop=True)
)

attractor_rows = []

for pred_id in range(NUM_CLASSES):
    predicted_count = int(cm_raw[:, pred_id].sum())
    correct = int(cm_raw[pred_id, pred_id])
    wrong_inflow = predicted_count - correct

    attractor_rows.append({
        "class_id": pred_id,
        "word": id_to_word[pred_id],
        "true_support": int(class_support[pred_id]),
        "predicted_count": predicted_count,
        "correct": correct,
        "wrong_inflow": wrong_inflow,
        "wrong_fraction_of_predictions": (
            wrong_inflow / predicted_count
            if predicted_count > 0 else 0.0
        ),
    })

prediction_attractors_df = (
    pd.DataFrame(attractor_rows)
    .sort_values(
        ["wrong_inflow", "wrong_fraction_of_predictions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Aliases useful for the final export cell.
baseline_directional_confusions = directional_confusions_df
baseline_pair_confusions = pair_confusions_df
baseline_prediction_attractors = prediction_attractors_df

print("=" * 88)
print("TOP DIRECTIONAL CONFUSIONS")
print("=" * 88)
display(directional_confusions_df.head(20).round(4))

print("\nTOP PAIR CONFUSIONS")
display(pair_confusions_df.head(20).round(4))

print("\nPREDICTION ATTRACTORS")
display(prediction_attractors_df.head(15).round(4))

DIRECTIONAL_PATH = MODEL_DIR / "01_word_level_baseline_directional_confusions.csv"
PAIR_PATH = MODEL_DIR / "01_word_level_baseline_pair_confusions.csv"
BIDIRECTIONAL_PATH = MODEL_DIR / "01_word_level_baseline_bidirectional_confusions.csv"
ATTRACTOR_PATH = MODEL_DIR / "01_word_level_baseline_prediction_attractors.csv"

directional_confusions_df.to_csv(DIRECTIONAL_PATH, index=False, encoding="utf-8-sig")
pair_confusions_df.to_csv(PAIR_PATH, index=False, encoding="utf-8-sig")
bidirectional_confusions_df.to_csv(BIDIRECTIONAL_PATH, index=False, encoding="utf-8-sig")
prediction_attractors_df.to_csv(ATTRACTOR_PATH, index=False, encoding="utf-8-sig")

print("\nSaved confusion tables.")


In [ ]:
# ============================================================
# CELL 15 : PREDICTION INSPECTION TABLE
# ============================================================

inspection_rows = []

for idx in range(len(test_targets)):
    true_id = int(test_targets[idx])
    pred_id = int(test_predictions[idx])

    order = np.argsort(-test_probabilities[idx])
    top3 = order[:3]
    true_rank = int(np.where(order == true_id)[0][0] + 1)

    row = {
        "dataset_index": idx,
        "true_class_id": true_id,
        "true_word": id_to_word[true_id],
        "pred_class_id": pred_id,
        "pred_word": id_to_word[pred_id],
        "correct": bool(true_id == pred_id),
        "confidence": float(test_confidences[idx]),
        "true_rank": true_rank,
        "true_in_top3": bool(true_rank <= 3),
        "top1_word": id_to_word[int(top3[0])],
        "top1_prob": float(test_probabilities[idx, top3[0]]),
        "top2_word": id_to_word[int(top3[1])],
        "top2_prob": float(test_probabilities[idx, top3[1]]),
        "top3_word": id_to_word[int(top3[2])],
        "top3_prob": float(test_probabilities[idx, top3[2]]),
    }

    manifest_row = test_df.iloc[idx]

    for col in [
        "video_path",
        "mouth_cache_path",
        "speaker_id",
        "segment_id",
        "word_start",
        "word_end",
        "clip_start",
        "clip_end",
    ]:
        if col in test_df.columns:
            row[col] = manifest_row[col]

    inspection_rows.append(row)

prediction_inspection_df = pd.DataFrame(inspection_rows)

wrong_top1_true_top3_df = prediction_inspection_df[
    (~prediction_inspection_df["correct"])
    & prediction_inspection_df["true_in_top3"]
].copy()

print("=" * 88)
print("PREDICTION INSPECTION")
print("=" * 88)
print("Total                   :", len(prediction_inspection_df))
print("Correct                 :", int(prediction_inspection_df["correct"].sum()))
print("Wrong                   :", int((~prediction_inspection_df["correct"]).sum()))
print("Overall true in Top-3   :", int(prediction_inspection_df["true_in_top3"].sum()))
print("Wrong Top-1 / true Top-3:", len(wrong_top1_true_top3_df))

print("\nHigh-confidence correct")
display(
    prediction_inspection_df[
        prediction_inspection_df["correct"]
    ]
    .sort_values("confidence", ascending=False)
    .head(10)
)

print("\nHigh-confidence wrong")
display(
    prediction_inspection_df[
        ~prediction_inspection_df["correct"]
    ]
    .sort_values("confidence", ascending=False)
    .head(10)
)

print("\nWrong Top-1 but true in Top-3")
display(
    wrong_top1_true_top3_df
    .sort_values("confidence", ascending=False)
    .head(20)
)

PREDICTION_INSPECTION_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_prediction_inspection.csv"
)

prediction_inspection_df.to_csv(
    PREDICTION_INSPECTION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", PREDICTION_INSPECTION_PATH)


In [ ]:
# ============================================================
# CELL 16 : FINAL EXPERIMENT EXPORT
# ============================================================

def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)

    if not path.exists():
        return None

    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


zero_recall_words = (
    class_report_df.loc[
        class_report_df["recall"] == 0,
        "word"
    ]
    .astype(str)
    .tolist()
)

top_directional = (
    directional_confusions_df.iloc[0]
    if len(directional_confusions_df) > 0
    else None
)

top_pair = (
    pair_confusions_df.iloc[0]
    if len(pair_confusions_df) > 0
    else None
)

top_attractor = (
    prediction_attractors_df.iloc[0]
    if len(prediction_attractors_df) > 0
    else None
)

best_epoch = checkpoint.get("epoch", EXP0_REFERENCE_BEST_EPOCH)

val_metrics_ckpt = checkpoint.get("val_metrics", {}) or {}

best_val_loss = float(
    val_metrics_ckpt.get("loss", np.nan)
)

best_val_accuracy = float(
    val_metrics_ckpt.get(
        "accuracy",
        EXP0_REFERENCE_VAL_ACCURACY
    )
)

best_val_macro_f1 = float(
    val_metrics_ckpt.get(
        "macro_f1",
        checkpoint.get(
            "best_val_macro_f1",
            EXP0_REFERENCE_VAL_MACRO_F1
        )
    )
)

experiment_record = {
    "project": "VocaLip Thai",
    "experiment_id": "EXP0",
    "experiment_name": EXPERIMENT_NAME,
    "task": "44-class isolated Thai visual word classification",
    "input_stream": "mouth-only",

    "dataset": {
        "manifest": str(MANIFEST_PATH),
        "total": int(len(df)),
        "train": int(len(train_df)),
        "val": int(len(val_df)),
        "test": int(len(test_df)),
        "classes": int(NUM_CLASSES),
        "excluded_subtitle_occluded_train_clips": 8,
    },

    "preprocessing": {
        "target_frames": TARGET_FRAMES,
        "roi_size": ROI_SIZE,
        "cache_root": str(CACHE_ROOT),
        "normalization": "ImageNet mean/std",
    },

    "model": {
        "architecture": (
            "FrameCNN -> 2-layer bidirectional GRU -> "
            "TemporalAttention -> 44-class classifier"
        ),
        "feature_dim": FEATURE_DIM,
        "gru_hidden": GRU_HIDDEN,
        "gru_layers": GRU_LAYERS,
        "dropout": MODEL_DROPOUT,
        "parameters": int(total_parameters),
    },

    "training_reference": {
        "loss": "weighted CrossEntropyLoss",
        "weighted_sampler": False,
        "optimizer": "AdamW",
        "initial_lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": EPOCHS,
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "grad_clip_norm": GRAD_CLIP_NORM,
    },

    "best_validation": {
        "loss": best_val_loss,
        "accuracy": best_val_accuracy,
        "macro_f1": best_val_macro_f1,
    },

    "final_test": {
        "loss": test_loss,
        "accuracy": test_accuracy,
        "macro_f1": test_macro_f1,
        "top3_accuracy": test_top3_accuracy,
        "correct": num_correct,
        "wrong": num_wrong,
        "wrong_top1_true_top3": wrong_true_in_top3,
        "wrong_top1_true_top3_rate": wrong_true_in_top3_rate,
    },

    "random_reference": {
        "top1": 1.0 / NUM_CLASSES,
        "top3": 3.0 / NUM_CLASSES,
    },

    "diagnostics": {
        "zero_recall_class_count": int(len(zero_recall_words)),
        "zero_recall_words": zero_recall_words,

        "strongest_directional": (
            {
                "true_word": str(top_directional["true_word"]),
                "pred_word": str(top_directional["pred_word"]),
                "count": int(top_directional["confusion_count"]),
                "rate": float(top_directional["confusion_rate"]),
            }
            if top_directional is not None else None
        ),

        "strongest_pair": (
            {
                "word_a": str(top_pair["word_a"]),
                "word_b": str(top_pair["word_b"]),
                "combined_errors": int(top_pair["pair_confusion_count"]),
            }
            if top_pair is not None else None
        ),

        "strongest_attractor": (
            {
                "word": str(top_attractor["word"]),
                "wrong_inflow": int(top_attractor["wrong_inflow"]),
            }
            if top_attractor is not None else None
        ),
    },

    "evaluation_scope": {
        "closed_vocabulary": True,
        "in_domain": True,
        "speaker_disjoint": False,
        "unseen_patient_generalization_claim": False,
    },

    "limitations": [
        "Current split is not strict speaker-disjoint evaluation.",
        "The result should be interpreted as an in-domain baseline.",
        "Several test classes have only 2-4 samples; per-class F1 is noisy.",
        "Thai lexical tone is not directly visible from mouth motion alone.",
        "No unknown/reject class is included.",
        "The test set has already been opened; do not tune EXP0 using test results.",
    ],

    "status": "BASELINE_FROZEN",
    "next_experiment": "Notebook 02 — Self-Supervised Visual Pretraining",

    "sha256": {
        "checkpoint": file_sha256(BEST_MODEL_PATH),
        "clean_manifest": file_sha256(MANIFEST_PATH),
        "label_map": (
            file_sha256(LABEL_MAP_PATH)
            if LABEL_MAP_PATH.exists()
            else None
        ),
    },
}

FINAL_JSON = MODEL_DIR / "01_word_level_baseline_final_experiment.json"
FINAL_METRICS_CSV = MODEL_DIR / "01_word_level_baseline_final_metrics.csv"

with open(FINAL_JSON, "w", encoding="utf-8") as f:
    json.dump(
        experiment_record,
        f,
        ensure_ascii=False,
        indent=2
    )

final_metrics_df = pd.DataFrame([{
    "experiment_id": EXPERIMENT_ID,
    "classes": NUM_CLASSES,
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
    "best_epoch": best_epoch,
    "val_accuracy": best_val_accuracy,
    "val_macro_f1": best_val_macro_f1,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_top3_accuracy": test_top3_accuracy,
    "correct": num_correct,
    "wrong": num_wrong,
    "wrong_top1_true_top3": wrong_true_in_top3,
    "wrong_top1_true_top3_rate": wrong_true_in_top3_rate,
}])

final_metrics_df.to_csv(
    FINAL_METRICS_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 88)
print("EXPERIMENT 0 FINAL RECORD")
print("=" * 88)
print(
    f"Dataset : {len(df)} | "
    f"train {len(train_df)} | "
    f"val {len(val_df)} | "
    f"test {len(test_df)} | "
    f"classes {NUM_CLASSES}"
)
print("Best epoch :", best_epoch)
print(
    f"Val Acc    : {best_val_accuracy:.2%} | "
    f"Macro F1 {best_val_macro_f1:.4f}"
)
print(
    f"Test Acc   : {test_accuracy:.2%} | "
    f"Macro F1 {test_macro_f1:.4f} | "
    f"Top-3 {test_top3_accuracy:.2%}"
)
print("Split scope: in-domain, NOT speaker-disjoint")
print("Status     : BASELINE FROZEN")
print("Next       : Notebook 02 — SSL Visual Pretraining")
print()
print("Saved JSON:", FINAL_JSON)
print("Saved CSV :", FINAL_METRICS_CSV)

baseline_experiment_record = experiment_record


In [ ]:
self.projection = nn.Sequential(
    nn.Flatten(),
    nn.Linear(128, feature_dim),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.10)
)

## Experiment 0 interpretation

เมื่อผล recovery ตรงกับผลเดิม:

- Top-1 Accuracy ≈ **26.25%**
- Macro F1 ≈ **0.2327**
- Top-3 Accuracy ≈ **48.26%**
- Random Top-1 = **2.27%**
- Random Top-3 = **6.82%**

Experiment 0 จึงเป็น baseline ที่แสดงว่ามี visual signal ที่โมเดลเรียนรู้ได้ แต่ยังไม่พร้อม deployment

**ห้ามใช้ test set ปรับ hyperparameter ต่อใน Experiment 0**

ขั้นถัดไปให้แยกเป็น Notebook 02: **Self-Supervised Visual Pretraining** เพื่อรักษา ablation ให้ชัดเจน
